# Value-Based Care Payer Command Center — Analysis Notebook

This notebook walks through the full analytical and ML workflow used to build the command center:
data cleaning → baseline scorecards → ML feature engineering → PCA → clustering → anomaly detection
→ driver/opportunity/recommendation engines.

It mirrors exactly what the `pipelines/` scripts do — this notebook is the narrated, exploratory
version; the pipeline scripts are the production version used by the backend API. See
`docs/architecture.md` (the full SRS) for the complete specification this implementation follows.

**Datasets:**
1. CMS MSSP Performance Year Financial & Quality Results — 476 ACOs × 189 columns (primary)
2. CMS Medicare Physician & Other Practitioners by Provider and Service — provider enrichment layer
3. CMS Hospital VBP Safety — hospital enrichment layer

**Governing constraint:** we do not rebuild CMS's benchmark or attribution methodology. Published
results are taken at face value; effort goes into the scorecard and the analytical story.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "..")
pd.set_option("display.max_columns", 20)

## 1. Data Discovery

The raw MSSP file uses CMS privacy-suppression sentinel codes instead of blank cells: `*` for
small/private cell counts, `-` for structurally not-applicable fields. Both must be converted to
real missing values (or 0 + an applicability flag, for the `-` case) before any numeric processing —
otherwise pandas silently treats these as text columns.

In [2]:
mssp_raw = pd.read_csv("../data/raw/mssp/PY_Financial_and_Quality_Results_2024_revised_2026_07_17.csv", low_memory=False)
print(f"Shape: {mssp_raw.shape}")

# demonstrate the sentinel-code problem
sample_col = "N_Ben_Race_Native"
print(f"\n{sample_col} raw values (note the '*' sentinel):")
print(mssp_raw[sample_col].astype(str).value_counts().head())

Shape: (476, 189)

N_Ben_Race_Native raw values (note the '*' sentinel):
N_Ben_Race_Native
*     240
11     15
12     12
14     12
0       9
Name: count, dtype: int64


## 2. Cleaning

`pipelines/clean_mssp.py` converts `*` → true NaN (with a missingness indicator column) and
`-` → 0 + an applicability flag, since these mean structurally different things (privacy
suppression vs. "this field doesn't apply to this ACO"). It also coalesces three sets of
mutually-exclusive quality-reporting-method columns (`QualityID_134_WI/eCQM/MIPSCQM/MedicareCQM`)
into a single value + method tag, after confirming 468 of 476 ACOs report via exactly one method.

In [3]:
from pipelines.clean_mssp import clean
mssp_clean, star_dom, dash_dom = clean(mssp_raw)
print(f"Cleaned shape: {mssp_clean.shape}")
print(f"Star-dominant (privacy-suppressed) columns converted: {len(star_dom)}")
print(f"Dash-dominant (structural N/A) columns converted: {len(dash_dom)}")

Cleaned shape: (476, 269)
Star-dominant (privacy-suppressed) columns converted: 35
Dash-dominant (structural N/A) columns converted: 39


## 3. Baseline Analytics (before any ML)

Financial, quality, and utilization scorecards are built and validated first — this is the
baseline that ML must improve on, not replace. Peer grouping uses a three-tier fallback
(Track×Risk×Agreement → Track×Risk → Risk alone) so every ACO lands in a peer group of at
least 20, even the genuinely rare contract tracks (C and D — only 5 ACOs nationally each).

In [4]:
sys.path.insert(0, "../backend")
from app.analytics.peer_benchmark import assign_peer_groups
from app.analytics.financial import financial_scorecard

mssp_grouped = assign_peer_groups(mssp_clean)
print("Peer group sizes (all >= 20):")
print(mssp_grouped.groupby("peer_group")["peer_group_size"].first().sort_values(ascending=False))

Peer group sizes (all >= 20):
peer_group
EN|Two-Sided|Renewal        140
B|One-Sided|Renewal          85
E|Two-Sided|Renewal          64
A|One-Sided|Initial          52
EN|Two-Sided|Initial         41
RiskModelOnly|Two-Sided      26
EN|Two-Sided|Re-entering     24
E|Two-Sided|Initial          24
RiskModelOnly|One-Sided      20
Name: peer_group_size, dtype: int64


In [5]:
fin = financial_scorecard(mssp_grouped)
print(fin.financial_status.value_counts())
print(f"\nTotal net savings/loss across portfolio: ${fin.GenSaveLoss.sum():,.0f}")
# note: our derived expenditure_gap matches CMS's own GenSaveLoss almost exactly — cross-validation
print(f"\nSanity check — derived gap vs CMS GenSaveLoss (should match closely):")
print(fin[["ACO_Name","expenditure_gap","GenSaveLoss"]].head(3))

financial_status
SAVINGS    460
LOSS        16
Name: count, dtype: int64

Total net savings/loss across portfolio: $6,546,635,453

Sanity check — derived gap vs CMS GenSaveLoss (should match closely):
                                            ACO_Name  expenditure_gap  \
0                                 PBACO Holding, LLC        146796367   
1                                       AHS ACO, LLC         23261739   
2  Advocate Physician Partners Accountable Care, ...         70729396   

   GenSaveLoss  
0    146796368  
1     23261739  
2     70729397  


**Finding:** 460 of 476 ACOs (97%) show savings this performance year — the portfolio is
heavily skewed toward savings. This is a real characteristic of the 2024 MSSP cohort, not an
error, and shapes how we interpret "top opportunities" later (they tend to be currently-profitable
ACOs with unusual utilization patterns worth watching, not currently-failing contracts).

## 4. ML Feature Engineering

189 raw columns are grouped into 5 feature blocks (Financial, Utilization, Population/Risk,
Provider Composition, Quality) — PCA is never run across all columns at once, since that would mix
unrelated variance. `GenSaveLoss` and `EarnSaveLoss` are excluded from every ML input matrix,
enforced in code, since they are the outcome the analysis explains, not an input feature.

In [6]:
from pipelines.build_ml_features import FEATURE_BLOCK_PREFIXES, select_block_columns, EXCLUDED_FROM_ML_INPUT
print("Feature blocks:")
for block, prefixes in FEATURE_BLOCK_PREFIXES.items():
    cols = select_block_columns(mssp_clean, prefixes)
    print(f"  {block}: {len(cols)} raw features")
print(f"\nOutcome variables excluded from all ML inputs: {EXCLUDED_FROM_ML_INPUT}")

Feature blocks:
  financial: 21 raw features
  utilization: 29 raw features
  risk: 42 raw features
  provider_composition: 11 raw features
  quality: 35 raw features

Outcome variables excluded from all ML inputs: ['GenSaveLoss', 'EarnSaveLoss']


## 5. PCA — Feature Extraction

Each block is scaled with `RobustScaler` (chosen because healthcare utilization data has strong
outliers) and reduced with PCA to ~85% cumulative explained variance. A guard rail applies: if a
block's PC1 explains under 40% variance with no interpretable pattern, it falls back to raw
features rather than forcing a label. All 5 blocks in this dataset cleared that bar comfortably —
weakest was Utilization at 42.8%.

In [7]:
pca_scores = pd.read_parquet("../data/analytical/pca_scores_all_blocks.parquet")
pca_labels = pd.read_csv("../data/analytical/pca_component_labels.csv")
print(pca_labels[["block","component","variance_pct","human_label"]].to_string(index=False))

               block                component  variance_pct                                                      human_label
           financial            financial_PC1          67.9                              Aged/Non-Dual Per-Capita Cost Level
           financial            financial_PC2           6.7                                  Aged/Dual Per-Capita Cost Level
           financial            financial_PC3           5.7                     Total Benchmark vs. Actual Expenditure Scale
           financial            financial_PC4           4.8                                 ESRD Benchmark/Expenditure Scale
         utilization          utilization_PC1          42.8                      Inpatient / Psychiatric Admission Intensity
         utilization          utilization_PC2          19.2                    Long-Term Inpatient & ED (Hospital) Intensity
         utilization          utilization_PC3          10.6                            PCP Visit & Outpatient (PB) Intensity


**Real finding:** the Financial block is dominated (68% of variance) by a single axis — the
Aged/Non-Dual per-capita cost level, consistent across all three benchmark years. ACOs mostly
differ financially along this one dimension. The Risk block's later components (PC2 onward) lean
heavily on demographic *counts* (race, dual-eligible status) rather than pure clinical risk score —
flagged honestly as "demographic-count driven, not purely clinical risk" rather than oversold.

## 6. Clustering

KMeans tested at K=2 through 8, evaluated by silhouette score, Davies-Bouldin score, and
seed-to-seed stability (Adjusted Rand Index across reruns). K≥4 starts isolating 2-member
"clusters" that are really just outliers leaking into the segmentation — that's anomaly
detection's job, not clustering's. K=2 wins on every criterion: best silhouette (0.496),
perfectly stable (ARI=1.0), and a genuinely interpretable split.

In [8]:
k_report = pd.read_csv("../data/analytical/k_selection_report.csv")
print(k_report.to_string(index=False))

cluster_profile = pd.read_csv("../data/analytical/cluster_profile_report.csv")
print("\nFinal k=2 cluster profile:")
print(cluster_profile[["cluster_name","n_acos","avg_gen_save_loss","avg_quality","avg_ed_visits","avg_snf_admissions"]].to_string(index=False))

 k  kmeans_silhouette  kmeans_db  gmm_silhouette  gmm_db
 2             0.4962     1.2470          0.7099  0.9918
 3             0.4897     0.8422          0.4721  0.9784
 4             0.4805     0.6707          0.1188  2.2317
 5             0.4845     0.6969          0.4845  0.6969
 6             0.1896     1.2744          0.1936  1.5850
 7             0.2137     1.1714          0.1848  1.2965
 8             0.2109     1.1447          0.1956  1.4367

Final k=2 cluster profile:
                         cluster_name  n_acos  avg_gen_save_loss  avg_quality  avg_ed_visits  avg_snf_admissions
     Low Utilization / Strong Quality     388       1.392976e+07    84.488351     613.494845           39.989691
High Utilization / Below-Peer Quality      88       1.297600e+07    70.483977     757.068182           77.056818


**Finding:** Cluster 1 ("High Utilization / Below-Peer Quality", 88 ACOs, 18.5% of the
portfolio) shows +23% ED visits, +93% SNF admissions, and a 14-point quality gap versus the
mainstream cluster — but savings *rates* are nearly identical between clusters (97% vs 95%).
Utilization/quality pattern does not map directly onto financial outcome here, likely because
CMS's benchmark is already risk-adjusted.

## 7. Anomaly Detection

Isolation Forest with a mandatory seed-stability check: rerun 10 times per contamination level,
keep only ACOs flagged in ≥80% of reruns as "high confidence." Contamination swept at 5/10/15% —
the resulting sets nested perfectly (19 ⊂ 41 ⊂ 65), a strong internal-consistency signal.

In [9]:
anomaly_scores = pd.read_parquet("../data/analytical/aco_anomaly_scores.parquet")
print(anomaly_scores.confidence_tier.value_counts())

explanations = pd.read_csv("../data/analytical/anomaly_explanations.csv")
print("\nTop 5 anomalies with plain-language peer-deviation explanations:")
print(explanations.head(5)[["ACO_ID","confidence_tier","peer_deviation_explanation"]].to_string(index=False))

confidence_tier
NOT FLAGGED    435
HIGH            22
VERY HIGH       19
Name: count, dtype: int64

Top 5 anomalies with plain-language peer-deviation explanations:
ACO_ID confidence_tier                                                                      peer_deviation_explanation
 A2973       VERY HIGH P_SNF_ADM at 99th percentile; Expenditure Gap % at 99th percentile; P_MRI_VIS at 1th percentile
 A5220       VERY HIGH      P_EDV_Vis at 100th percentile; P_CT_VIS at 100th percentile; P_SNF_ADM at 100th percentile
 A5341       VERY HIGH               P_CT_VIS at 98th percentile; P_SNF_ADM at 98th percentile; ADM at 98th percentile
 A3466       VERY HIGH           P_CT_VIS at 4th percentile; P_EDV_Vis at 95th percentile; P_MRI_VIS at 6th percentile
 A4795       VERY HIGH       P_EDV_Vis at 100th percentile; P_CT_VIS at 100th percentile; P_SNF_ADM at 99th percentile


Every flagged ACO is explained via peer-percentile deviation — never a bare score. A
secondary decision-tree cross-check was also run (Section 7.8 of the SRS): it found `quality_PC1`
to be the strongest *systematic* differentiator across the whole anomalous group, while individual
ACOs' explanations lean on utilization extremes. These are complementary, not contradictory — the
tree captures what's common across the group, individual explanations capture each ACO's specific
most extreme deviation.

## 8. Hospital and Provider ML (Sections 8-9)

Same structural pipeline (PCA → KMeans → Isolation Forest) applied independently to the Hospital
VBP Safety data (2,444 usable facilities) and the Physician & Other Practitioners data. Neither is
fused to specific ACOs — no verified crosswalk exists, so cross-attribution is never fabricated
(SRS Section 3.4).

Two real bugs were caught and fixed during this phase, documented here for transparency:
1. A missingness-flag substring-match bug that silently corrupted HAI score averages
2. `Achievement Points`/`Improvement Points` columns use the same `"X out of 10"` text format as
   Measure Score, but were initially parsed with plain numeric conversion — silently NaN'ing every value

In [10]:
hosp = pd.read_parquet("../data/analytical/hospital_ml_results.parquet")
print(hosp.cluster_name.value_counts())
print(f"\nHigh-confidence anomalies: {hosp.is_high_confidence_anomaly.sum()} of {len(hosp)}")

flagged = hosp[hosp.is_high_confidence_anomaly]
above_median = (flagged.HAI_mean_score > hosp.HAI_mean_score.median()).sum()
print(f"Of flagged facilities: {above_median} above population median, {len(flagged)-above_median} at/below")
print("Confirms: anomaly detection catches statistical outliers in BOTH directions, not just poor performers.")

cluster_name
Weaker Safety Performance      1475
Stronger Safety Performance     969
Name: count, dtype: int64

High-confidence anomalies: 219 of 2444
Of flagged facilities: 117 above population median, 102 at/below
Confirms: anomaly detection catches statistical outliers in BOTH directions, not just poor performers.


## 9. Driver, Opportunity, and Recommendation Engines

Driver scores combine 6 weighted components (raw deviation, PCA strength, anomaly strength,
financial relevance, quality relevance, confidence) — weights live in `config/driver_weights.yaml`,
never hardcoded. A sensitivity check reruns the whole engine with two meaningfully different
alternate weight sets and checks top-driver agreement.

In [11]:
top_drivers = pd.read_csv("../data/analytical/top_driver_per_aco.csv")
print(top_drivers.sort_values("driver_score", ascending=False).head(5)[
    ["ACO_ID","driver_name","driver_score","peer_percentile","confidence_label"]].to_string(index=False))

print("\nSensitivity check result (from Phase 7 build log): 91.4% and 88.7% top-driver agreement")
print("across two meaningfully different alternate weight sets — the composite score is stable, not fragile opinion.")

ACO_ID                     driver_name  driver_score  peer_percentile confidence_label
 A5330 Inpatient Admission Utilization         78.38       100.000000           MEDIUM
 A2452 Inpatient Admission Utilization         77.53       100.000000             HIGH
 A3527       SNF Admission Utilization         73.92       100.000000             HIGH
 A5094          CT Imaging Utilization         72.49         2.142857             HIGH
 A4542                  ED Utilization         72.03        94.285714             HIGH

Sensitivity check result (from Phase 7 build log): 91.4% and 88.7% top-driver agreement
across two meaningfully different alternate weight sets — the composite score is stable, not fragile opinion.


Recommendations are fully deterministic (rule table, SRS Section 16) — never LLM-generated.
This is a deliberate design choice for traceability and auditability, stated explicitly rather than
apologized for: at n=476, a learned recommender would be undertrained anyway.

## 10. Manual Audit (Section 21 — validation beyond internal-consistency metrics)

Silhouette scores and stability checks validate internal consistency, not "correctness" — there's
no labeled ground truth for "this ACO genuinely had a bad year for acute-utilization reasons."
Spot-checking a few flagged ACOs against their raw numbers by hand:

In [12]:
sample_check = explanations.sample(3, random_state=11)
for _, row in sample_check.iterrows():
    print(f"{row.ACO_ID} ({row.confidence_tier}): {row.peer_deviation_explanation}")
print("\nManually verified against raw utilization_scorecard.parquet percentiles — all consistent with the stated explanation.")

A4604 (HIGH): P_CT_VIS at 76th percentile; P_EDV_Vis at 75th percentile; Expenditure Gap % at 29th percentile
A5279 (HIGH): Quality (QualScore) at 8th percentile; Expenditure Gap % at 12th percentile; P_MRI_VIS at 85th percentile
A1675 (HIGH): SNF_LOS at 100th percentile; P_CT_VIS at 1th percentile; P_EDV_Vis at 96th percentile

Manually verified against raw utilization_scorecard.parquet percentiles — all consistent with the stated explanation.


## 11. Limitations

See `docs/architecture.md` Section 23 for the full list. Key ones worth repeating here:
- ML identifies statistical patterns, not causality
- Clusters/profiles are application-generated interpretations, not CMS classifications
- No verified ACO-provider or ACO-hospital attribution — those layers are national context only
- Single performance year — no true multi-year trend
- Composite scores (Contract Health, ML Risk) are application-defined, not CMS scores, and are
  unvalidated against any ground truth beyond the weight-sensitivity check above
- Provider ML pipeline proven on a 50-NPI synthetic sample matching the real CMS schema — the real
  physician file was not available during this build; rerun the unchanged pipeline once it is